# 01 — Environment Setup

Get Ollama running, pull a model, install the `ai_tools` packages, verify everything works. Two paths: local (if you have a discrete GPU) or Colab (if you don't). The course targets local — Colab is the fallback.

## Path A: Local setup

### 1. Create a virtual environment

```bash
python3 -m venv ~/ai-env
source ~/ai-env/bin/activate   # or ~/ai-env/Scripts/activate on Windows
```

Every command from here on assumes the venv is active.

### 2. Install Ollama

Ollama is the model runtime. One-line install:

```bash
# Linux / macOS
curl -fsSL https://ollama.com/install.sh | sh

# Windows: download installer from https://ollama.com/download
```

Start the daemon:

```bash
ollama serve   # runs in foreground — leave it running in a terminal
```

Or on Linux, if you installed via the script, it started automatically as a systemd service.

### 3. Pull models

Pick based on your VRAM. You need at least one model for the course; having two enables the cross-model critique pattern from notebook 00.

| VRAM | Primary model | Optional second model | Command |
|---|---|---|---|
| 8 GB | `qwen3:8b` | `gemma3:4b` | `ollama pull qwen3:8b && ollama pull gemma3:4b` |
| 16 GB | `qwen3.6:35b-a3b` | `qwen3:8b` | `ollama pull qwen3.6:35b-a3b && ollama pull qwen3:8b` |
| 24 GB | `qwen3.6:27b` | `gemma4:9b` | `ollama pull qwen3.6:27b && ollama pull gemma4:9b` |

Pull time: 5-15 minutes per model depending on your connection. Models download once and persist.

```bash
ollama list   # verify the pull worked
```

### 4. Install `ai_tools` packages

Clone the repo and install all packages in editable mode:

```bash
cd ~  # or wherever you keep projects
git clone https://github.com/jdean314159/ai_tools.git
cd ai_tools
make install
```

`make install` creates the venv if it doesn't exist and installs the suite packages (`llm_harness_core`, `llm_engines`, `engram`, `llm_inspector`, `rag_lib`, `agent_lib`, `llm_inspector_ui`, and `language_tutor`) into the project environment. This takes a few minutes.

Install Jupyter:

```bash
pip install jupyter
```

### 5. Verify

Run this script to confirm Ollama responds, the model is loaded, and the toolkit packages import:

In [5]:
# verify_setup.py — save this, run it from your terminal
import sys
import requests

def check(desc, test):
    try:
        test()
        print(f"✓ {desc}")
        return True
    except Exception as e:
        print(f"✗ {desc}: {e}")
        return False

def ollama_reachable():
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json()["models"]]
    if not models:
        raise RuntimeError("Ollama running but no models pulled")
    print(f"  Models: {', '.join(models)}")

def model_responds():
    r = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={
            "model": "qwen3:8b",  # adjust to your pulled model
            "messages": [{"role": "user", "content": "say ok"}],
            "temperature": 0.0,
        },
        timeout=60,
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    if "ok" not in content.lower():
        raise RuntimeError(f"Model responded but output unexpected: {content}")

def packages_importable():
    for pkg in ["llm_harness_core", "llm_engines", "engram", "llm_inspector", "rag_lib", "agent_lib"]:
        __import__(pkg)

if __name__ == "__main__":
    passed = 0
    passed += check("Ollama reachable", ollama_reachable)
    passed += check("Model responds", model_responds)
    passed += check("ai_tools packages installed", packages_importable)
    print(f"\n{passed}/3 checks passed.")

  Models: qwen3.6:27b, gemma3:4b, qwen3:8b
✓ Ollama reachable
✓ Model responds
✓ ai_tools packages installed

3/3 checks passed.


Save that as `verify_setup.py` and run:

```bash
python verify_setup.py
```

All three checks should pass. If they don't, see troubleshooting below.

### 6. Launch Jupyter

```bash
cd ~/ai_tools/course/notebooks
jupyter notebook
```

Open `00_llm_fundamentals.ipynb` and run the first cell to verify everything works end-to-end.

## Path B: Google Colab

Colab provides a free T4 GPU (15 GB VRAM). It runs in your browser — zero local install. Constraints:

- Session timeout: 12 hours max, 90 minutes idle
- Storage: lost on disconnect (mount Google Drive for persistence)
- Models fit: `qwen3:8b` and smaller

Setup cell (run once per session):

In [ ]:
# Colab setup cell (working version)
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, time
os.environ['PATH'] = f"/usr/local/bin:{os.environ['PATH']}"

!pkill -9 ollama 2>/dev/null
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(10)

!ollama pull qwen3:8b
!ollama pull gemma3:4b

# Clone or upload the course repository before running this cell.
from pathlib import Path
requirements = Path('requirements.txt')
if not requirements.exists():  # compatibility while course/ still lives in ai_tools
    requirements = Path('course/requirements.txt')
%pip install -q -r {requirements}

In [ ]:
# Verify (Colab)
import requests

r = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "qwen3:8b", 
        "messages": [{"role": "user", "content": "say ok"}],
        "reasoning_effort": "none",  # critical - disables thinking mode
        "max_tokens": 10
    },
    timeout=60,
)
assert r.status_code == 200, f"Request failed: {r.status_code}"
print("✓ Ollama responding")

import llm_engines, engram, llm_inspector
print("✓ Packages installed")

print("\nColab setup complete. You can now run notebook 00.")

**Colab-specific notes:**

- API keys: Use Colab Secrets (🔑 icon in left sidebar), not hardcoded strings. Access via `from google.colab import userdata; key = userdata.get('ANTHROPIC_API_KEY')`.
- Persistence: Mount Drive with `from google.colab import drive; drive.mount('/content/drive')` and save work to `/content/drive/MyDrive/ai_course/`.
- Model changes: `!ollama pull MODEL_NAME` works mid-session. Daemon persists until disconnect.

## Troubleshooting

### Ollama not reachable

**Symptom:** `verify_setup.py` fails on "Ollama reachable."

**Fix:**

```bash
# Check if ollama is running
curl http://localhost:11434/api/tags

# If connection refused, start it
ollama serve

# On Linux, if installed via script, check systemd
systemctl status ollama
```

### Model pull fails

**Symptom:** `ollama pull` hangs or times out.

**Fix:** Models are large (5-15 GB). Ensure stable connection and disk space. Retry the pull — Ollama resumes interrupted downloads.

### Packages not found

**Symptom:** `verify_setup.py` fails on "ai_tools packages installed."

**Fix:**

```bash
# Confirm venv is active
which python   # should show ~/ai-env/bin/python

# Reinstall
cd ~/ai_tools
make install

# If make install failed, check the error. Common: missing build tools.
# On Ubuntu: sudo apt install python3-dev build-essential
```

### Model responds but output is wrong

**Symptom:** Model returns valid JSON but content doesn't make sense.

**Fix:** You probably have a different model pulled than the one hardcoded in `verify_setup.py`. Edit the script to match your pulled model name exactly (check `ollama list`).

### If all else fails
Copy the code and errors from the Colab page, and ask Claude, ChatGPT or another enterprise AI to determine why it did not work for you.

## What you have now

- A working LLM runtime (Ollama) with at least one model
- The `ai_tools` packages installed and importable
- Jupyter running locally or a Colab session ready

API keys are optional at this point. Every notebook through 07 works with Ollama alone. Notebooks 08-09 optionally use cloud APIs for comparison, but fall back to local if the key isn't set.

**Security note:** When you do add API keys, use environment variables or Colab Secrets — never hardcode them in notebooks you might commit. The `.gitignore` in `ai_tools` already excludes `.env` files; use `python-dotenv` to load them.

---
**Next:** [02 — Engine Basics](02_engine_basics.ipynb) introduces `llm_engines`, which abstracts the `chat()` function you wrote in notebook 00 into something production-ready.